# TAMBO TR Detector Layout Optimization — v5 (Evolutionary Pruning)

Evolutionary algorithm that starts with 10,000 candidate detectors on the Colca Valley
mountain surface and prunes down to 90, using gradient-saliency fitness scoring via a
DeepSets (permutation-invariant) reconstruction network.

See `CLAUDE.md` for full design documentation.

In [ ]:
# Standard library imports
import torch
import torch._utils
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import copy
import warnings
import functools
from pathlib import Path
from matplotlib import colors, animation
from matplotlib.colors import LogNorm, Normalize
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import HTML
import os
import gc

# v3 modules (via sys.path injection in modules_v5/__init__.py)
import modules_v5
from modules.generate_showers   import GenerateShowers
from modules.shower_computation  import ComputeShowerDetection
from modules.detector_response   import GetCounts_differentiable, SmearN, TimeAverage_vectorized
from modules.reconstruction      import NormalizeLabels, DenormalizeLabels, EarlyStopping
from modules.utility_functions   import reconstructability, U_PR, U_E, U_angle

# v4 modules
from modules_v4.tr_geometry      import load_tr_mountain
from modules_v4.tr_surface_map   import SurfaceEastMap
from modules_v4.tr_plane_kernel  import GetCounts_planeaware

# v5 modules
from modules_v5.ev_deepsets      import DeepSetsReconstruction
from modules_v5.ev_population    import Population, build_input_batch
from modules_v5.ev_selection     import compute_detector_fitness, prune_weakest, mutate_positions

output_dir = "./outputs/NN_Files_EV_v5"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/Python_Layout", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generate_new_data = True
print("device:", device)

In [ ]:
# Physics constants (identical to v4)
c0 = 0.29979           # Speed of light [m/ns]
theta_max = np.pi * 65 / 180
log_01 = torch.tensor([np.log(0.1)], dtype=torch.float32)
log_10 = torch.tensor([np.log(10)], dtype=torch.float32)

# Tank values
IntegrationWindow = 128.0   # ns
sigma_time        = 10.0    # ns
TankArea          = 6.859 * np.pi
TankRadius        = np.sqrt(6.859)
R_min             = 2.0

# Background
Bgr_e_per_m2 = 0.000000200 * IntegrationWindow
fluxB_e = torch.tensor([TankArea * Bgr_e_per_m2])

# Optimization sizes — v5 evolutionary
N_units_init  = 10000    # starting detector count
N_units_final = 90       # target final count (same as v4)
RelResCounts  = 0.05
NUM_FEATURES  = 7        # [x=N, y=Up, z=z_cont, N_int, T_int, x0, y0]

# NN training sizes
Nevents     = 20000
Nvalidation = 2000
Ntest       = 300
Nbatch      = 500        # showers per fitness evaluation

# Evolutionary algorithm parameters
n_generations = 30

# Geometry
GEOMETRY_PATH = "../../TAMBOSim/resources/basic_geometry.h5"
GROUP    = "colca_valley_30000"
DET_KEY  = "detector1"
N_PLANES = 24
EAST_ENTRY    = 1500.0   # correct calibration (v4 active scripts + v6)
LAYER_EAST_DX =  150.0   # correct calibration (v4 active scripts + v6)

In [ ]:
mountain = load_tr_mountain(GEOMETRY_PATH, GROUP, DET_KEY,
                            east_entry=EAST_ENTRY, layer_east_dx=LAYER_EAST_DX, n_planes=N_PLANES)
surface  = SurfaceEastMap.from_mountain(mountain, grid_h=256, grid_w=256).to(device)

print(f"Mountain: N=[{mountain.n_min:.0f}, {mountain.n_max:.0f}] m  "
      f"Up=[{mountain.u_min:.0f}, {mountain.u_max:.0f}] m  "
      f"East=[{mountain.east_lo:.0f}, {mountain.east_hi:.0f}] m")
print(f"z_cont range on mountain surface: [0, {mountain.east_to_z_cont(mountain.east_lo):.1f}]")

In [ ]:
%%time
population = Population.initial(mountain, surface, n_units=N_units_init, device=device)
print(f"Initial population: {population.size} detectors")
print(f"z_cont: [{population.z_cont.min():.2f}, {population.z_cont.max():.2f}]")

# Save initial layout
population.save_layout(f"{output_dir}/Python_Layout/Layout_0.txt")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

N_mtn  = mountain.centroids_NUE[:, 0]
Up_mtn = mountain.centroids_NUE[:, 1]
East_mtn = mountain.centroids_NUE[:, 2]

sc0 = axes[0].scatter(N_mtn, Up_mtn, c=East_mtn, s=1, cmap='terrain', alpha=0.3)
plt.colorbar(sc0, ax=axes[0], label='East [m]')
axes[0].scatter(population.x.cpu(), population.y.cpu(), c='red', s=1, alpha=0.3, label=f'Detectors ({population.size})')
axes[0].set_xlabel('North [m]'); axes[0].set_ylabel('Up [m]')
axes[0].set_title('Initial detector layout (top-down)'); axes[0].legend()

axes[1].hist(population.z_cont.cpu().numpy(), bins=30, color='steelblue', edgecolor='k')
axes[1].set_xlabel('z_cont (plane index)'); axes[1].set_ylabel('count')
axes[1].set_title('Initial z_cont distribution')
plt.tight_layout(); plt.show()

In [ ]:
generate_showers_instance = GenerateShowers(output_dir=output_dir, device=device, batch_size=60)

In [ ]:
# Partial-apply physics constants into v3's post-processing callables
_SmearN = functools.partial(SmearN, RelResCounts=RelResCounts)
_TimeAverage = functools.partial(TimeAverage_vectorized,
                                 IntegrationWindow=IntegrationWindow,
                                 sigma_time=sigma_time)

# Shower vertical recenter (same as v4)
Y_TARGET = mountain.u_min + 0.25 * (mountain.u_max - mountain.u_min)
_y_shift_state = {"value": None}


def _apply_y_shift(samples):
    """Shift samples[:, :, 1] (Up) by a cached offset.  Compute it on first call."""
    if _y_shift_state["value"] is None:
        with torch.no_grad():
            mask = samples[..., 3] > 0
            if mask.any():
                ys = samples[..., 1][mask]
                es = samples[..., 3][mask]
                centroid = float((ys * es).sum() / es.sum())
                _y_shift_state["value"] = float(Y_TARGET) - centroid
                print(f"[generate_showers] Y_SHIFT cached: centroid={centroid:.0f} m  "
                      f"shift={_y_shift_state['value']:+.0f} m  -> target={Y_TARGET:.0f} m")
            else:
                _y_shift_state["value"] = 0.0
    shift = _y_shift_state["value"]
    if shift == 0.0:
        return samples
    samples = samples.clone()
    samples[..., 1] = samples[..., 1] + shift
    return samples


def generate_showers(x_det, y_det, z_cont, log=False, number_of_showers=1,
                     device=device, use_cache=False):
    """Generate showers and compute plane-aware detector counts.

    z_cont : (n_det,) continuous plane index derived from the mountain surface map.
    """
    def _get_counts_planeaware(samples, x_d, y_d):
        samples = _apply_y_shift(samples)
        return GetCounts_planeaware(
            samples, x_d, y_d, z_cont,
            SmearN_fn=_SmearN,
            fluxB_e=fluxB_e.to(samples.device),
            TimeAverage_vectorized_fn=_TimeAverage,
        )

    out = ComputeShowerDetection(
        x_det, y_det,
        generate_showers_instance,
        _get_counts_planeaware,
        log=log,
        number_of_showers=number_of_showers,
        device=device,
        use_cache=use_cache,
        output_dir=output_dir,
        filter_plane=None,
    )

    # Patch Y0 to match the y-shifted samples
    shift = _y_shift_state["value"]
    if shift is not None and shift != 0.0:
        out = list(out)
        out[3] = out[3] + shift   # Y0
        out = tuple(out)
    return out

## Phase 1: Generate training data & train DeepSets NN

Use the initial 10k-detector layout to generate 20k training events.
The NN is trained with **random mask-dropout** (10–90% of detectors zeroed
per batch) so it learns robust per-detector representations.

In [ ]:
%%time

if generate_new_data:
    with torch.no_grad():
        N, T, X0, Y0, energy, sin_z, cos_z, sin_a, cos_a, _ = generate_showers(
            population.x, population.y, population.z_cont,
            log=False, number_of_showers=Nevents, use_cache=True
        )

    th = torch.atan2(sin_z, cos_z)
    ph = torch.atan2(sin_a, cos_a)
    E_norm, theta_norm, phi_norm = NormalizeLabels(energy, th, ph)

    inputs = build_input_batch(population, N, T, X0, Y0)
    labels = torch.stack([E_norm, theta_norm, phi_norm], dim=1).float()

    torch.save(inputs, f"{output_dir}/inputs.pt")
    torch.save(labels, f"{output_dir}/labels.pt")
    print(f"Training data: inputs {tuple(inputs.shape)}, labels {tuple(labels.shape)}")

In [ ]:
%%time

if generate_new_data:
    with torch.no_grad():
        N, T, X0, Y0, energy, sin_z, cos_z, sin_a, cos_a, _ = generate_showers(
            population.x, population.y, population.z_cont,
            log=False, number_of_showers=Nvalidation, use_cache=True
        )

    th = torch.atan2(sin_z, cos_z)
    ph = torch.atan2(sin_a, cos_a)
    E_norm, theta_norm, phi_norm = NormalizeLabels(energy, th, ph)

    inputs_val = build_input_batch(population, N, T, X0, Y0)
    labels_val = torch.stack([E_norm, theta_norm, phi_norm], dim=1).float()

    torch.save(inputs_val, f"{output_dir}/inputs_val.pt")
    torch.save(labels_val, f"{output_dir}/labels_val.pt")

In [ ]:
if generate_new_data:
    with torch.no_grad():
        N, T, X0, Y0, energy, sin_z, cos_z, sin_a, cos_a, _ = generate_showers(
            population.x, population.y, population.z_cont,
            log=False, number_of_showers=Ntest, use_cache=True
        )

    th = torch.atan2(sin_z, cos_z)
    ph = torch.atan2(sin_a, cos_a)
    E_norm, theta_norm, phi_norm = NormalizeLabels(energy, th, ph)

    inputs_test = build_input_batch(population, N, T, X0, Y0)
    labels_test = torch.stack([E_norm, theta_norm, phi_norm], dim=1).float()

    torch.save(inputs_test, f"{output_dir}/inputs_test.pt")
    torch.save(labels_test, f"{output_dir}/labels_test.pt")

In [ ]:
inputs      = torch.load(f"{output_dir}/inputs.pt")
labels      = torch.load(f"{output_dir}/labels.pt")
inputs_val  = torch.load(f"{output_dir}/inputs_val.pt")
labels_val  = torch.load(f"{output_dir}/labels_val.pt")
inputs_test = torch.load(f"{output_dir}/inputs_test.pt")
labels_test = torch.load(f"{output_dir}/labels_test.pt")
print("inputs:", inputs.shape, "  labels:", labels.shape)

In [ ]:
# Per-feature normalization (frozen for the rest of the run)
input_mean = inputs.mean(dim=(0, 1))   # (7,)
input_std  = inputs.std(dim=(0, 1))    # (7,)
input_std[input_std < 1e-8] = 1.0
print("input_mean:", input_mean)
print("input_std: ", input_std)

In [ ]:
model = DeepSetsReconstruction(
    input_features=NUM_FEATURES,
    embedding_dim=64,
    hidden_dim=128,
    output_dim=3,
).to(device)

criterion = nn.MSELoss()
optimizer_nn = torch.optim.Adam(model.parameters(), lr=3e-5)
print("DeepSets parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
%%time

# Normalize training / val / test data
inputs_norm      = (inputs      - input_mean) / input_std
inputs_val_norm  = (inputs_val  - input_mean) / input_std
inputs_test_norm = (inputs_test - input_mean) / input_std

dataset    = TensorDataset(inputs_norm, labels)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=False, num_workers=0)

val_losses = []
losses = []
early_stopper = EarlyStopping(patience=30)

for epoch_nn in range(300):
    model.train()
    epoch_loss = 0
    total_batch = 0

    for batch_inputs, batch_labels in dataloader:
        bx = batch_inputs.to(device)    # (B, N, 7)
        by = batch_labels.to(device)    # (B, 3)

        # Random mask-dropout: zero out 10–90% of detectors per event
        B_b, N_b, _ = bx.shape
        drop_rate = torch.rand(1).item() * 0.8 + 0.1   # uniform [0.1, 0.9]
        mask = (torch.rand(B_b, N_b, device=device) > drop_rate).float()

        outputs = model(bx, mask=mask)
        loss = criterion(outputs, by)
        epoch_loss += loss.item()
        total_batch += 1
        loss.backward()
        optimizer_nn.step()
        optimizer_nn.zero_grad()

    # Validation (no mask dropout — full layout)
    model.eval()
    with torch.no_grad():
        val_out = model(inputs_val_norm.to(device))
        val_loss = criterion(val_out, labels_val.to(device))
    val_losses.append(val_loss.item())
    losses.append(epoch_loss / total_batch)

    early_stopper(val_loss)
    if early_stopper.early_stop:
        print(f"Early stopping at epoch {epoch_nn}")
        break

    if (epoch_nn + 1) % 50 == 0:
        print(f"Epoch {epoch_nn+1}  train_loss={losses[-1]:.5f}  val_loss={val_loss:.5f}")

plt.plot(losses,     color="blue", label="Training Loss")
plt.plot(val_losses, color="red",  label="Validation Loss")
plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid()
plt.title("DeepSets Training (with mask-dropout)"); plt.tight_layout(); plt.show()

torch.save(model.state_dict(), f"{output_dir}/model_weights.pth")
print("NN weights saved")

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(inputs_test_norm.to(device)).cpu()

E_pred, theta_pred, phi_pred = DenormalizeLabels(outputs[:, 0], outputs[:, 1], outputs[:, 2])
E_lb,  theta_lb,  phi_lb  = DenormalizeLabels(labels_test[:, 0], labels_test[:, 1], labels_test[:, 2])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(E_lb.cpu(), E_pred.cpu(), alpha=.5, color="r")
axes[0].plot([0, E_lb.max().cpu()], [0, E_lb.max().cpu()], "k-", lw=2)
axes[0].set_xlabel("True E"); axes[0].set_ylabel("Predicted E"); axes[0].grid()

axes[1].scatter(theta_lb.cpu(), theta_pred.cpu(), alpha=.5, color="r")
axes[1].plot([theta_lb.min().cpu(), theta_lb.max().cpu()], [theta_lb.min().cpu(), theta_lb.max().cpu()], "k-", lw=2)
axes[1].set_xlabel("True Theta"); axes[1].set_ylabel("Predicted Theta"); axes[1].grid()

axes[2].scatter(phi_lb.cpu(), phi_pred.cpu(), alpha=.5, color="r")
axes[2].plot([phi_lb.min().cpu(), phi_lb.max().cpu()], [phi_lb.min().cpu(), phi_lb.max().cpu()], "k-", lw=2)
axes[2].set_xlabel("True Phi"); axes[2].set_ylabel("Predicted Phi"); axes[2].grid()

plt.suptitle("DeepSets Reconstruction Quality (initial 10k layout)", fontsize=13)
plt.tight_layout(); plt.show()

## Phase 2: Evolutionary Pruning Loop

Geometric decay from 10,000 → 90 over ~30 generations.
Each generation: score → prune → mutate → (optional NN fine-tune) → save.

In [ ]:
# Move normalization tensors to device
input_mean_d = input_mean.to(device)
input_std_d  = input_std.to(device)

# Geometric schedule: 10000 → 90 over n_generations steps
schedule = np.round(np.geomspace(N_units_init, N_units_final, n_generations + 1)).astype(int)
# Ensure exact start and end
schedule[0]  = N_units_init
schedule[-1] = N_units_final
# Ensure strictly decreasing
for i in range(1, len(schedule)):
    if schedule[i] >= schedule[i-1]:
        schedule[i] = schedule[i-1] - 1

# Mutation sigma decays over generations (explore broadly early, refine later)
sigma_mut_schedule = np.linspace(80.0, 10.0, n_generations)

print("Pruning schedule:")
for i, s in enumerate(schedule):
    print(f"  gen {i:2d}: {s} detectors")

In [ ]:
%%time

fitness_history = []
det_count_history = [population.size]

for gen in range(n_generations):
    target = int(schedule[gen + 1])
    sigma_mut = float(sigma_mut_schedule[gen])

    # 1. Score every detector via gradient saliency
    fitness_scores, U_now = compute_detector_fitness(
        population, model, generate_showers,
        n_samples=Nbatch,
        input_mean=input_mean_d,
        input_std=input_std_d,
    )
    fitness_history.append(U_now)

    print(f"Gen {gen:2d}  N_det={population.size:5d} -> {target:5d}  "
          f"U={U_now:.4f}  fitness: [{fitness_scores.min():.3e}, {fitness_scores.max():.3e}]  "
          f"sigma_mut={sigma_mut:.1f}")

    # 2. Prune weakest detectors to target size
    prune_weakest(population, fitness_scores, target_size=target)
    det_count_history.append(population.size)

    # 3. Mutate survivors (exploration)
    mutate_positions(population, sigma=sigma_mut, frac=0.5)

    # 4. Fine-tune NN every 5 generations
    if (gen + 1) % 5 == 0:
        print(f"  Fine-tuning NN at gen {gen+1}...")
        model.train()
        opt_ft = torch.optim.Adam(model.parameters(), lr=1e-5)
        with torch.no_grad():
            N_ft, T_ft, X0_ft, Y0_ft, e_ft, sz, cz, sa, ca, _ = generate_showers(
                population.x, population.y, population.z_cont,
                log=False, number_of_showers=min(2500, Nbatch * 3), use_cache=True)
            th_ft = torch.atan2(sz, cz); ph_ft = torch.atan2(sa, ca)
            e_n, th_n, ph_n = NormalizeLabels(e_ft, th_ft, ph_ft)
            inp_ft = build_input_batch(population, N_ft, T_ft, X0_ft, Y0_ft)
            inp_ft = (inp_ft - input_mean_d) / input_std_d
            lab_ft = torch.stack([e_n, th_n, ph_n], dim=1).float().to(device)

        # Apply random mask-dropout during fine-tuning too
        ft_ds = TensorDataset(inp_ft, lab_ft)
        ft_dl = DataLoader(ft_ds, batch_size=64, shuffle=True, drop_last=True)
        for ft_ep in range(3):
            for bx, by in ft_dl:
                bx, by = bx.to(device), by.to(device)
                B_b, N_b, _ = bx.shape
                drop_rate = torch.rand(1).item() * 0.8 + 0.1
                mk = (torch.rand(B_b, N_b, device=device) > drop_rate).float()
                pred = model(bx, mask=mk)
                loss = criterion(pred, by)
                loss.backward()
                opt_ft.step(); opt_ft.zero_grad()
        model.eval()

    # 5. Save layout + utility
    population.save_layout(f"{output_dir}/Python_Layout/Layout_{gen+1}.txt")
    np.savetxt(f"{output_dir}/Python_Layout/Utilities.txt", np.array(fitness_history))

    # 6. Checkpoint every 5 generations
    if (gen + 1) % 5 == 0:
        torch.save({
            "model_state_dict": model.state_dict(),
            "generation": gen + 1,
            "detector_count": population.size,
            "fitness_history": fitness_history,
        }, f"{output_dir}/checkpoint.pth")

print(f"\nEvolution complete: {population.size} detectors remain")

## Phase 3: Final NN retrain on the 90-detector layout

In [ ]:
%%time

print(f"Retraining NN on final {population.size}-detector layout...")

with torch.no_grad():
    N_final, T_final, X0_f, Y0_f, e_final, sz, cz, sa, ca, _ = generate_showers(
        population.x, population.y, population.z_cont,
        log=False, number_of_showers=Nevents, use_cache=True)
    th_f = torch.atan2(sz, cz); ph_f = torch.atan2(sa, ca)
    e_n, th_n, ph_n = NormalizeLabels(e_final, th_f, ph_f)
    inputs_final = build_input_batch(population, N_final, T_final, X0_f, Y0_f)
    labels_final = torch.stack([e_n, th_n, ph_n], dim=1).float()

# Recompute normalization for the final layout
input_mean_final = inputs_final.mean(dim=(0, 1))
input_std_final  = inputs_final.std(dim=(0, 1))
input_std_final[input_std_final < 1e-8] = 1.0

inputs_final_norm = (inputs_final - input_mean_final) / input_std_final

final_ds = TensorDataset(inputs_final_norm, labels_final)
final_dl = DataLoader(final_ds, batch_size=256, shuffle=True, drop_last=False)

model_final = DeepSetsReconstruction(input_features=NUM_FEATURES, embedding_dim=64, hidden_dim=128).to(device)
opt_final = torch.optim.Adam(model_final.parameters(), lr=3e-5)
early_final = EarlyStopping(patience=30)

# Generate val data at final layout
with torch.no_grad():
    Nv, Tv, X0v, Y0v, ev, szv, czv, sav, cav, _ = generate_showers(
        population.x, population.y, population.z_cont,
        log=False, number_of_showers=Nvalidation, use_cache=True)
    thv = torch.atan2(szv, czv); phv = torch.atan2(sav, cav)
    en_v, thn_v, phn_v = NormalizeLabels(ev, thv, phv)
    inp_val_f = build_input_batch(population, Nv, Tv, X0v, Y0v)
    lab_val_f = torch.stack([en_v, thn_v, phn_v], dim=1).float()
    inp_val_f_norm = (inp_val_f - input_mean_final) / input_std_final

final_losses = []
final_val_losses = []
for ep in range(300):
    model_final.train()
    ep_loss = 0; n_b = 0
    for bx, by in final_dl:
        bx, by = bx.to(device), by.to(device)
        B_b, N_b, _ = bx.shape
        # Mask dropout at final size too
        drop_rate = torch.rand(1).item() * 0.5 + 0.1
        mk = (torch.rand(B_b, N_b, device=device) > drop_rate).float()
        out = model_final(bx, mask=mk)
        loss = criterion(out, by)
        ep_loss += loss.item(); n_b += 1
        loss.backward(); opt_final.step(); opt_final.zero_grad()

    model_final.eval()
    with torch.no_grad():
        v_out = model_final(inp_val_f_norm.to(device))
        v_loss = criterion(v_out, lab_val_f.to(device))
    final_losses.append(ep_loss / n_b)
    final_val_losses.append(v_loss.item())
    early_final(v_loss)
    if early_final.early_stop:
        print(f"Early stop at epoch {ep}"); break
    if (ep + 1) % 50 == 0:
        print(f"Epoch {ep+1} train={final_losses[-1]:.5f} val={v_loss:.5f}")

torch.save(model_final.state_dict(), f"{output_dir}/model_weights.pth")
torch.save({
    "model_state_dict": model_final.state_dict(),
    "generation": n_generations,
    "detector_count": population.size,
    "fitness_history": fitness_history,
    "input_mean": input_mean_final,
    "input_std": input_std_final,
}, f"{output_dir}/checkpoint.pth")

plt.plot(final_losses,     color="blue", label="Train")
plt.plot(final_val_losses, color="red",  label="Val")
plt.legend(); plt.grid(); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Final NN Training (90 detectors)"); plt.tight_layout(); plt.show()

## Phase 4: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fitness_history, 'ko-', ms=4, lw=1)
axes[0].set_xlabel('Generation'); axes[0].set_ylabel('Utility U')
axes[0].set_title('Fitness per generation'); axes[0].grid()

axes[1].plot(det_count_history, 'bs-', ms=4, lw=1)
axes[1].set_xlabel('Generation'); axes[1].set_ylabel('Detector count')
axes[1].set_title('Population size per generation'); axes[1].grid()
axes[1].set_yscale('log')

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
N_mtn  = mountain.centroids_NUE[:, 0]
Up_mtn = mountain.centroids_NUE[:, 1]
East_mtn = mountain.centroids_NUE[:, 2]

sc_bg = ax.scatter(N_mtn, Up_mtn, c=East_mtn, s=1, cmap='terrain', alpha=0.4)
plt.colorbar(sc_bg, ax=ax, label='East [m]')

# Load initial layout for comparison
lay0 = np.loadtxt(f'{output_dir}/Python_Layout/Layout_0.txt')
ax.scatter(lay0[:, 0], lay0[:, 1], c='blue', s=1, alpha=0.15, label=f'Initial ({len(lay0)})')
ax.scatter(population.x.cpu(), population.y.cpu(), c='red', s=30, alpha=0.8, label=f'Final ({population.size})')
ax.set_xlabel('North [m]'); ax.set_ylabel('Up [m]')
ax.set_title('Detector layout: initial (blue) vs final (red)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax3 = fig.add_subplot(111, projection='3d')

ax3.scatter(N_mtn, East_mtn, Up_mtn, c=East_mtn, cmap='terrain',
            s=1, alpha=0.15, depthshade=True)

with torch.no_grad():
    east_final = surface(population.x, population.y).cpu().numpy()
    zc_final = population.z_cont.cpu().numpy()

sc_det = ax3.scatter(
    population.x.cpu().numpy(), east_final, population.y.cpu().numpy(),
    c=zc_final, cmap='plasma', vmin=0, vmax=N_PLANES - 1,
    s=50, zorder=5, depthshade=False)
plt.colorbar(sc_det, ax=ax3, label='z_cont (plane index)', pad=0.1)

ax3.view_init(elev=20, azim=30)
ax3.set_xlabel('North [m]'); ax3.set_ylabel('East [m]'); ax3.set_zlabel('Up [m]')
ax3.set_title('Final detector positions on the Colca Valley mountain surface (v5 EA)')
plt.tight_layout(); plt.show()

In [ ]:
%%capture

layout_dir = Path(f'{output_dir}/Python_Layout')
layout_files = sorted(layout_dir.glob('Layout_*.txt'),
                      key=lambda p: int(p.stem.split('_')[1]))

# Load all layouts (variable row count is fine with np.loadtxt)
layouts = [np.loadtxt(f) for f in layout_files]
print(f"Loaded {len(layouts)} layout frames")

N_mtn_a  = mountain.centroids_NUE[:, 0]
Up_mtn_a = mountain.centroids_NUE[:, 1]
East_mtn_a = mountain.centroids_NUE[:, 2]

fig_a = plt.figure(figsize=(11, 7))
ax_a  = fig_a.add_subplot(111, projection='3d')

ax_a.scatter(N_mtn_a, East_mtn_a, Up_mtn_a, c=East_mtn_a, cmap='terrain',
             s=1, alpha=0.15, depthshade=True)
ax_a.set_xlabel('North [m]'); ax_a.set_ylabel('East [m]'); ax_a.set_zlabel('Up [m]')
ax_a.view_init(elev=20, azim=30)

lay0 = layouts[0]
N_d0, Up_d0, zc_d0 = lay0[:, 0], lay0[:, 1], lay0[:, 2]
East_d0 = mountain.east_entry - zc_d0 * mountain.layer_east_dx
det_scatter = ax_a.scatter(N_d0, East_d0, Up_d0, c=zc_d0, cmap='plasma',
                           vmin=0, vmax=N_PLANES - 1, s=5, zorder=5, depthshade=False)
title_obj = ax_a.set_title(f'Gen 0  ({len(lay0)} detectors)')


def _update(frame):
    lay = layouts[frame]
    N_d, Up_d, zc_d = lay[:, 0], lay[:, 1], lay[:, 2]
    East_d = mountain.east_entry - zc_d * mountain.layer_east_dx
    det_scatter._offsets3d = (N_d, East_d, Up_d)
    det_scatter.set_array(zc_d)
    # Scale point size with detector count (larger when fewer)
    s = max(5, min(50, int(3000 / len(lay))))
    det_scatter.set_sizes(np.full(len(lay), s))
    title_obj.set_text(f'Gen {frame}  ({len(lay)} detectors)')
    return det_scatter, title_obj


anim = animation.FuncAnimation(fig_a, _update, frames=len(layouts), interval=300, blit=False)

gif_path = Path(output_dir) / 'layout_evolution_3d.gif'
anim.save(str(gif_path), writer='pillow', fps=4, dpi=100)
print(f"GIF saved to {gif_path}")

In [ ]:
HTML(anim.to_jshtml())

In [ ]:
# Save final utility history
np.savetxt(f'{output_dir}/Python_Layout/Utilities.txt', np.array(fitness_history))
print(f"Final utility: {fitness_history[-1]:.4f}")
print(f"Final detector count: {population.size}")
print(f"Layout files saved: Layout_0.txt through Layout_{n_generations}.txt")